# SSDU 데모 — 모집단(population) 자기지도 복원

**SSDU** = **여러 스캔(train split)으로 하나의 네트워크**를 자기지도 학습한다 (GT 없음).
- 슬라이스마다 획득 마스크 Ω를 **Θ(데이터 일관성) / Λ(loss)** 로 분할
- Θ만 보고 복원 → **Λ 위치의 측정 k-space**를 맞히도록 학습 (MixL1L2 k-space loss)
- 검증은 val split에서 RSS로 monitoring (학습엔 GT 안 씀)

**ZS-SSL과 차이**: ZS-SSL은 *단일 스캔*을 추론 시점에 학습, SSDU는 *train 전체*로 미리 학습해 test에 적용.

In [ ]:
import os, sys, time, glob
sys.path.insert(0, '/home/sonwonjun/research/MRRecon/code')
import numpy as np, torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from mrrecon.config import Config
from mrrecon.data.loaders import list_slice_files, read_slice
from mrrecon.data.datasets import SSDUDataset
from mrrecon.data.masks import undersampling_mask
from mrrecon.models import build_unrolled
from mrrecon.losses import MixL1L2Loss
from mrrecon.metrics import rss_metrics, match_scale
from mrrecon.core.common import center_crop
from mrrecon.core.inference import recon_unrolled

# small config + train subset so the demo runs in a few minutes
cfg=Config(); cfg.tissue='knee'; cfg.device='cuda' if torch.cuda.is_available() else 'cpu'
cfg.model='ssdu'; cfg.nb_unroll_blocks=4; cfg.res_blocks=3; cfg.cg_iter=4; cfg.mu=0.05
cfg.acc_rate=4; cfg.acs_lines=24; cfg.mask_type='vds_lustig'; cfg.rho=0.4
cfg.lr=1e-3; cfg.epochs=6; cfg.crop_size=320; cfg.seed=1234
DEMO_TRAIN=120     # train slices for the demo (전체는 7049)
dev=torch.device(cfg.device); print('device',dev)

train_files=list_slice_files(cfg.data_root,cfg.tissue,'train',-1,'',False)[:DEMO_TRAIN]
val_files  =list_slice_files(cfg.data_root,cfg.tissue,'val',  -1,'',False)[:10]
test_files =list_slice_files(cfg.data_root,cfg.tissue,'test', -1,'',False)
print(f'train(demo)={len(train_files)}  val(monitor)={len(val_files)}  test={len(test_files)}')

## 1. Θ/Λ 분할 (한 슬라이스 예시)
SSDU는 매 슬라이스의 Ω를 Θ(DC)/Λ(loss)로 나눈다 (`SSDUMaskSplitter`, Gaussian, rho=0.4).

In [ ]:
ds_train=SSDUDataset(cfg, train_files, train=True)
b=ds_train[0]
theta=b['trn_mask'][0].numpy(); lam=b['loss_mask'][0].numpy()
line=lambda m:m[m.shape[0]//2]
fig,ax=plt.subplots(1,2,figsize=(13,4))
ax[0].plot(line(theta),label='Theta (DC)'); ax[0].plot(line(lam),label='Lambda (loss)',alpha=0.7)
ax[0].set_title('Omega -> Theta / Lambda (1D phase)'); ax[0].legend(); ax[0].set_xlabel('phase column'); ax[0].set_ylim(-0.1,1.1)
ax[1].imshow(np.stack([line(theta),line(lam)]),aspect='auto',cmap='gray',interpolation='nearest')
ax[1].set_yticks([0,1]); ax[1].set_yticklabels(['Theta','Lambda']); ax[1].set_xlabel('phase column'); ax[1].set_title('which columns')
plt.tight_layout(); plt.show()
print(f'Theta={int(line(theta).sum())}  Lambda={int(line(lam).sum())}  (둘 합 = Omega)')

## 2. 모델 + 데이터로더

In [ ]:
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
dl=DataLoader(ds_train,batch_size=1,shuffle=True,num_workers=0)
model=build_unrolled(cfg).to(dev); opt=torch.optim.Adam(model.parameters(),lr=cfg.lr); loss_fn=MixL1L2Loss().to(dev)
print(f'model={cfg.model} params={sum(p.numel() for p in model.parameters())/1e3:.1f}k | train slices={len(ds_train)}')

## 3. SSDU 학습 루프 (train 전체를 순회)
ZS-SSL과 달리 **여러 슬라이스**를 돈다. 각 슬라이스: Θ/Λ k-space loss. 매 epoch 끝에 val(RSS) monitoring.

In [ ]:
def to_dev(b): return (b['x_in'].to(dev),b['sens_maps'].to(dev),b['ref_kspace'].to(dev),b['trn_mask'].to(dev),b['loss_mask'].to(dev))
@torch.no_grad()
def val_metrics():
    model.eval(); ss=[]
    for i,fp in enumerate(val_files):
        k,s,r=read_slice(fp,crop_size=cfg.crop_size)
        om=undersampling_mask(k.shape[1:],cfg.acc_rate,cfg.acs_lines,cfg.mask_type,rng=np.random.default_rng(i+1),vds_power=cfg.vds_power)
        _,_,rec=recon_unrolled(model,k,s,om,dev); ss.append(rss_metrics(r,rec,crop_fn=center_crop)['ssim'])
    return float(np.mean(ss))
hist=[]; t0=time.time()
for ep in range(cfg.epochs):
    model.train(); el=0;n=0
    for b in dl:
        x,s,rk,tr,lm=to_dev(b); _,_,nw=model(x,s,tr,lm); loss=loss_fn(nw,rk)
        opt.zero_grad(); loss.backward(); opt.step(); el+=loss.item(); n+=1
    el/=n; vs=val_metrics(); hist.append({'ep':ep+1,'train':el,'val_ssim':vs})
    print(f'epoch {ep+1}/{cfg.epochs}  train_loss={el:.4f}  val_SSIM={vs:.4f}  ({time.time()-t0:.0f}s)')
print(f'done in {time.time()-t0:.0f}s')

In [ ]:
ep=[h['ep'] for h in hist]
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(ep,[h['train'] for h in hist],'o-'); ax[0].set_title('train k-space loss (Lambda)'); ax[0].set_xlabel('epoch'); ax[0].grid(alpha=0.3)
ax[1].plot(ep,[h['val_ssim'] for h in hist],'o-',color='C2'); ax[1].set_title('val SSIM vs RSS (monitor)'); ax[1].set_xlabel('epoch'); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. 학습된 모델을 test 슬라이스에 적용 → GT 비교
SSDU는 한 번 학습하면 **임의의 test 스캔에 바로 적용**(추가 학습 없이).

In [ ]:
model.eval()
tf=test_files[len(test_files)//2]
k,s,r=read_slice(tf,crop_size=cfg.crop_size)
om=undersampling_mask(k.shape[1:],cfg.acc_rate,cfg.acs_lines,cfg.mask_type,rng=np.random.default_rng(7),vds_power=cfg.vds_power)
ref,zf,rec=recon_unrolled(model,k,s,om,dev); m=rss_metrics(r,rec,crop_fn=center_crop)
rc=center_crop(np.abs(rec)); rr=center_crop(r); zfc=center_crop(np.abs(zf))
fig,ax=plt.subplots(1,3,figsize=(15,5))
ax[0].imshow(rr,cmap='gray',vmax=0.6*rr.max()); ax[0].set_title('RSS GT'); ax[0].axis('off')
ax[1].imshow(zfc,cmap='gray',vmax=0.6*zfc.max()); ax[1].set_title('zero-filled'); ax[1].axis('off')
ax[2].imshow(match_scale(rr,rc),cmap='gray',vmax=0.6*rr.max()); ax[2].set_title(f'SSDU recon\nSSIM={m["ssim"]:.3f} PSNR={m["psnr"]:.2f}'); ax[2].axis('off')
plt.suptitle(f'SSDU on test slice {tf.split(chr(47))[-1]}'); plt.tight_layout(); plt.show()
print(f'test SSDU: SSIM={m["ssim"]:.4f} PSNR={m["psnr"]:.3f} NMSE={m["nmse"]:.5f}')

**요약**
- SSDU = **train 전체로 하나의 네트워크 학습** (슬라이스마다 Θ/Λ k-space loss, GT 없음)
- 학습 후 **임의 test 스캔에 즉시 적용** (ZS-SSL은 스캔마다 재학습)
- 이 데모는 train 120장·6 epoch·작은 net (실제는 7049장·50 epoch 권장)
- 실제 실행: `python -m mrrecon.self_supervised --algo ssdu --tissue knee`